# Agent Debate & Consensus — Multiple Perspectives to Better Answers

## What This Notebook Teaches You

In Notebook 17, we saw two agents talking. Now we scale to **N agents** debating a topic, with a **judge** evaluating arguments and reaching a conclusion. This is the **multi-agent debate** pattern, inspired by research showing that debate among LLM agents improves reasoning accuracy.

In this notebook, you will:

1. **Understand the debate pattern** and its research foundations (Du et al. 2023)
2. **Build a DebateArena** — register N debaters + a judge, run round-based debates
3. **Build DebaterAgent** — agents that argue from assigned perspectives with structured arguments
4. **Build JudgeAgent** — evaluates arguments, scores debaters, declares conclusions
5. **Experiment 1**: Factual debate — "Is nuclear energy good for climate change?"
6. **Experiment 2**: Problem-solving — urban traffic congestion
7. **Experiment 3**: Code review — performance, security, maintainability angles
8. **Compare voting mechanisms** — majority, weighted, ranked vs judge
9. **Analyze scaling** — 2 vs 3 vs 5 debaters

---

> **Prerequisites**: Notebook 17 (multi-agent conversation).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Debate Pattern

### Research Foundations

**Du et al. (2023), "Improving Factuality and Reasoning in Language Models through Multiagent Debate"** showed that when multiple LLM instances debate their answers over several rounds, the final consensus is significantly more accurate than individual responses — especially on math and reasoning tasks.

### Why Debate Works

1. **Diverse initial answers** — different "agents" (or different prompts) generate different initial responses
2. **Error correction** — agents identify flaws in each other's reasoning
3. **Evidence aggregation** — each round adds new evidence and counter-arguments
4. **Convergence to truth** — incorrect positions tend to collapse under scrutiny

### The Architecture

```
                    ┌──────────┐
                    │  Judge   │  ← Evaluates all arguments
                    └────┬─────┘
                         │
         ┌───────────────┼───────────────┐
         ▼               ▼               ▼
   ┌──────────┐   ┌──────────┐   ┌──────────┐
   │Debater 1 │   │Debater 2 │   │Debater 3 │
   │(Pro)      │   │(Con)     │   │(Nuanced) │
   └──────────┘   └──────────┘   └──────────┘

Round 1: Each debater states position
Round 2: Debaters respond to each other
Round 3: Final arguments
Judge:   Scores arguments, declares conclusion
```

## 2. Building the Debate Infrastructure

### 2.1 — Structured Arguments

In [ ]:
@dataclass
class Argument:
    """A structured debate argument."""
    debater: str
    round_num: int
    claim: str
    evidence: str
    rebuttal: str = ""     # Response to opposing arguments
    confidence: float = 0.8

    def to_text(self):
        parts = [f"**Claim**: {self.claim}", f"**Evidence**: {self.evidence}"]
        if self.rebuttal:
            parts.append(f"**Rebuttal**: {self.rebuttal}")
        return "\n".join(parts)

    def __str__(self):
        return f"[{self.debater}, R{self.round_num}] {self.claim[:80]}..."


@dataclass
class JudgmentScore:
    """Score for a debater from the judge."""
    debater: str
    evidence_quality: float    # 0-10
    reasoning_quality: float   # 0-10
    rebuttal_quality: float    # 0-10
    overall: float = 0.0

    def compute_overall(self):
        self.overall = round(
            0.4 * self.evidence_quality +
            0.35 * self.reasoning_quality +
            0.25 * self.rebuttal_quality, 2
        )
        return self.overall

print("Argument and JudgmentScore dataclasses ready")

### 2.2 — DebaterAgent

In [ ]:
class DebaterAgent:
    """An agent that argues from a given perspective in a debate."""

    def __init__(self, name: str, perspective: str, system_prompt: str = ""):
        self.name = name
        self.perspective = perspective
        self.system_prompt = system_prompt or (
            f"You are '{name}', a debater arguing from the perspective: {perspective}. "
            f"Make structured arguments with clear claims and evidence. "
            f"When responding to other debaters, address their specific points. "
            f"Be persuasive but honest — acknowledge strong opposing points. "
            f"Keep each response to 4-6 sentences."
        )
        self.arguments: List[Argument] = []

    def make_argument(self, topic: str, round_num: int,
                       previous_arguments: List[Argument] = None) -> Argument:
        """Generate a structured argument for this round."""
        messages = [{"role": "system", "content": self.system_prompt}]

        if round_num == 1:
            messages.append({"role": "user", "content": (
                f"Debate topic: {topic}\n\n"
                f"This is Round 1. State your initial position clearly.\n"
                f"Format your response as:\n"
                f"CLAIM: [your main claim in 1-2 sentences]\n"
                f"EVIDENCE: [supporting evidence and reasoning in 2-3 sentences]"
            )})
        else:
            # Include previous arguments for context
            prev_text = "\n\n".join([
                f"{a.debater} argued:\n{a.to_text()}"
                for a in (previous_arguments or [])
                if a.debater != self.name  # Don't include own previous arguments
            ])

            messages.append({"role": "user", "content": (
                f"Debate topic: {topic}\n\n"
                f"Previous arguments from other debaters:\n{prev_text}\n\n"
                f"This is Round {round_num}. Respond to the other debaters' arguments.\n"
                f"Format your response as:\n"
                f"CLAIM: [your updated claim, considering their arguments]\n"
                f"EVIDENCE: [supporting evidence]\n"
                f"REBUTTAL: [your response to their specific points]"
            )})

        response = generate(messages, max_new_tokens=300)

        # Parse structured response
        claim = evidence = rebuttal = ""
        for line in response.split("\n"):
            line_stripped = line.strip()
            upper = line_stripped.upper()
            if upper.startswith("CLAIM:"):
                claim = line_stripped[6:].strip()
            elif upper.startswith("EVIDENCE:"):
                evidence = line_stripped[9:].strip()
            elif upper.startswith("REBUTTAL:"):
                rebuttal = line_stripped[9:].strip()

        # Fallback if parsing fails
        if not claim:
            claim = response[:200]
        if not evidence:
            evidence = response[200:] if len(response) > 200 else "See claim above."

        arg = Argument(
            debater=self.name,
            round_num=round_num,
            claim=claim,
            evidence=evidence,
            rebuttal=rebuttal
        )
        self.arguments.append(arg)
        return arg

    def reset(self):
        self.arguments = []

print("DebaterAgent class ready")

### 2.3 — JudgeAgent

In [ ]:
class JudgeAgent:
    """Evaluates debate arguments and reaches a conclusion."""

    def __init__(self):
        self.system_prompt = (
            "You are an impartial judge evaluating a debate. "
            "Score each debater on evidence quality (0-10), reasoning quality (0-10), "
            "and rebuttal quality (0-10). Be fair and specific in your evaluation. "
            "Then state your conclusion on the debate topic."
        )

    def evaluate(self, topic: str, all_arguments: List[Argument],
                  debater_names: List[str]) -> Dict:
        """Evaluate all arguments and produce scores + conclusion."""
        # Format all arguments for the judge
        args_text = ""
        for debater in debater_names:
            debater_args = [a for a in all_arguments if a.debater == debater]
            args_text += f"\n{'='*40}\n{debater}:\n{'='*40}\n"
            for a in debater_args:
                args_text += f"\nRound {a.round_num}:\n{a.to_text()}\n"

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": (
                f"Debate topic: {topic}\n\n"
                f"Arguments:{args_text}\n\n"
                f"Evaluate each debater. For each, provide:\n"
                f"DEBATER: [name]\n"
                f"EVIDENCE_SCORE: [0-10]\n"
                f"REASONING_SCORE: [0-10]\n"
                f"REBUTTAL_SCORE: [0-10]\n\n"
                f"Then provide:\n"
                f"CONCLUSION: [your overall conclusion on the debate topic]\n"
                f"WINNER: [name of the most persuasive debater]"
            )}
        ]

        response = generate(messages, max_new_tokens=600)

        # Parse scores
        scores = {}
        current_debater = None
        conclusion = ""
        winner = ""

        for line in response.split("\n"):
            line_stripped = line.strip()
            upper = line_stripped.upper()

            if upper.startswith("DEBATER:"):
                current_debater = line_stripped[8:].strip()
            elif upper.startswith("EVIDENCE_SCORE:") and current_debater:
                try:
                    val = float(re.search(r'[\d.]+', line_stripped[15:]).group())
                    scores.setdefault(current_debater, {})["evidence"] = min(val, 10)
                except (AttributeError, ValueError):
                    scores.setdefault(current_debater, {})["evidence"] = 5.0
            elif upper.startswith("REASONING_SCORE:") and current_debater:
                try:
                    val = float(re.search(r'[\d.]+', line_stripped[16:]).group())
                    scores.setdefault(current_debater, {})["reasoning"] = min(val, 10)
                except (AttributeError, ValueError):
                    scores.setdefault(current_debater, {})["reasoning"] = 5.0
            elif upper.startswith("REBUTTAL_SCORE:") and current_debater:
                try:
                    val = float(re.search(r'[\d.]+', line_stripped[15:]).group())
                    scores.setdefault(current_debater, {})["rebuttal"] = min(val, 10)
                except (AttributeError, ValueError):
                    scores.setdefault(current_debater, {})["rebuttal"] = 5.0
            elif upper.startswith("CONCLUSION:"):
                conclusion = line_stripped[11:].strip()
            elif upper.startswith("WINNER:"):
                winner = line_stripped[7:].strip()

        # Build JudgmentScore objects
        judgment_scores = []
        for debater in debater_names:
            s = scores.get(debater, {})
            js = JudgmentScore(
                debater=debater,
                evidence_quality=s.get("evidence", 5.0),
                reasoning_quality=s.get("reasoning", 5.0),
                rebuttal_quality=s.get("rebuttal", 5.0)
            )
            js.compute_overall()
            judgment_scores.append(js)

        return {
            "scores": judgment_scores,
            "conclusion": conclusion or response[-300:],
            "winner": winner,
            "raw_evaluation": response
        }

print("JudgeAgent class ready")

## 3. DebateArena — Orchestrating Multi-Agent Debates

In [ ]:
class DebateArena:
    """Orchestrates a multi-round debate between N debaters and a judge."""

    def __init__(self, topic: str, num_rounds: int = 3):
        self.topic = topic
        self.num_rounds = num_rounds
        self.debaters: List[DebaterAgent] = []
        self.judge = JudgeAgent()
        self.all_arguments: List[Argument] = []
        self.result = None

    def add_debater(self, debater: DebaterAgent):
        """Register a debater."""
        self.debaters.append(debater)

    def run(self, verbose=True) -> Dict:
        """Run the full debate."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"DEBATE: {self.topic}")
            print(f"Debaters: {', '.join(d.name for d in self.debaters)}")
            print(f"Rounds: {self.num_rounds}")
            print(f"{'='*70}")

        start_time = time.time()
        self.all_arguments = []

        # Reset all debaters
        for d in self.debaters:
            d.reset()

        # Run debate rounds
        for round_num in range(1, self.num_rounds + 1):
            if verbose:
                print(f"\n--- Round {round_num} ---")

            round_args = []
            for debater in self.debaters:
                # Each debater sees previous round's arguments from others
                prev_args = [a for a in self.all_arguments if a.round_num == round_num - 1] if round_num > 1 else []

                arg = debater.make_argument(self.topic, round_num, prev_args)
                round_args.append(arg)

                if verbose:
                    print(f"\n  [{debater.name}]")
                    print(f"  Claim: {arg.claim[:150]}{'...' if len(arg.claim) > 150 else ''}")
                    if arg.rebuttal:
                        print(f"  Rebuttal: {arg.rebuttal[:120]}{'...' if len(arg.rebuttal) > 120 else ''}")

            self.all_arguments.extend(round_args)

        # Judge evaluation
        if verbose:
            print(f"\n--- Judge Evaluation ---")

        debater_names = [d.name for d in self.debaters]
        judgment = self.judge.evaluate(self.topic, self.all_arguments, debater_names)

        elapsed = time.time() - start_time

        if verbose:
            print(f"\nScores:")
            for score in judgment['scores']:
                print(f"  {score.debater:20s}  evidence={score.evidence_quality:.1f}  "
                      f"reasoning={score.reasoning_quality:.1f}  "
                      f"rebuttal={score.rebuttal_quality:.1f}  "
                      f"OVERALL={score.overall:.2f}")
            print(f"\nWinner: {judgment['winner']}")
            print(f"Conclusion: {judgment['conclusion'][:300]}")
            print(f"\nTime: {elapsed:.1f}s")

        self.result = {
            "topic": self.topic,
            "num_debaters": len(self.debaters),
            "num_rounds": self.num_rounds,
            "total_arguments": len(self.all_arguments),
            "scores": judgment['scores'],
            "conclusion": judgment['conclusion'],
            "winner": judgment['winner'],
            "elapsed_seconds": round(elapsed, 1),
            "total_words": sum(len(a.claim.split()) + len(a.evidence.split()) + len(a.rebuttal.split())
                              for a in self.all_arguments)
        }
        return self.result

print("DebateArena class ready")

## 4. Experiment 1: Factual Debate — Nuclear Energy and Climate

Three debaters argue about whether nuclear energy is beneficial for addressing climate change. Each brings a different perspective.

In [ ]:
arena1 = DebateArena("Is nuclear energy good for addressing climate change?", num_rounds=3)

arena1.add_debater(DebaterAgent(
    name="Pro-Nuclear",
    perspective="Nuclear energy is essential for climate change mitigation",
    system_prompt=(
        "You are 'Pro-Nuclear', arguing that nuclear energy is essential for addressing "
        "climate change. Cite data on carbon emissions, energy density, reliability, and "
        "capacity factors. Reference countries like France that use nuclear successfully. "
        "Keep each response to 4-6 sentences. Use specific numbers when possible."
    )
))

arena1.add_debater(DebaterAgent(
    name="Anti-Nuclear",
    perspective="Nuclear energy is too risky and expensive; renewables are better",
    system_prompt=(
        "You are 'Anti-Nuclear', arguing that nuclear energy is too risky and expensive "
        "compared to renewables. Cite data on construction costs, waste storage challenges, "
        "accident risks (Chernobyl, Fukushima), and the falling costs of solar and wind. "
        "Keep each response to 4-6 sentences. Use specific numbers when possible."
    )
))

arena1.add_debater(DebaterAgent(
    name="Pragmatist",
    perspective="Nuclear has a role alongside renewables but needs reform",
    system_prompt=(
        "You are 'Pragmatist', arguing that the nuclear vs renewables debate is a false "
        "dichotomy. Both are needed. Nuclear provides baseload power that intermittent "
        "renewables cannot. But the nuclear industry needs regulatory reform, new reactor "
        "designs (SMRs), and better waste solutions. "
        "Keep each response to 4-6 sentences."
    )
))

result_e1 = arena1.run()

## 5. Experiment 2: Problem-Solving — Urban Traffic Congestion

Three expert perspectives collaborate to solve a real-world problem.

In [ ]:
arena2 = DebateArena("How should cities solve urban traffic congestion?", num_rounds=3)

arena2.add_debater(DebaterAgent(
    name="UrbanPlanner",
    perspective="Redesign cities around public transit and walkability",
    system_prompt=(
        "You are 'UrbanPlanner', arguing that traffic congestion is fundamentally a "
        "land-use problem. Solutions include transit-oriented development, mixed-use zoning, "
        "bike infrastructure, and pedestrian-friendly design. Reference cities like Amsterdam, "
        "Tokyo, and Copenhagen. Keep each response to 4-6 sentences."
    )
))

arena2.add_debater(DebaterAgent(
    name="TechOptimist",
    perspective="Technology like autonomous vehicles and smart traffic systems will solve congestion",
    system_prompt=(
        "You are 'TechOptimist', arguing that technology is the most practical solution. "
        "Smart traffic signals, congestion pricing, ride-sharing, autonomous vehicles, and "
        "AI-optimized routing can dramatically reduce congestion without expensive infrastructure. "
        "Keep each response to 4-6 sentences."
    )
))

arena2.add_debater(DebaterAgent(
    name="Economist",
    perspective="Price road usage correctly through congestion pricing and remove parking subsidies",
    system_prompt=(
        "You are 'Economist', arguing that congestion is caused by mispriced road access. "
        "Congestion pricing (like London, Stockholm, Singapore), removing free parking mandates, "
        "and road usage charges would reduce demand efficiently. The market, not planning or "
        "technology alone, should allocate scarce road space. Keep each response to 4-6 sentences."
    )
))

result_e2 = arena2.run()

## 6. Experiment 3: Code Review — Three Angles

Three reviewers examine a code design from different quality angles.

In [ ]:
code_to_review = """
def process_user_data(raw_data):
    users = []
    for item in raw_data:
        user = {}
        user['name'] = item['name'].strip()
        user['email'] = item['email'].lower().strip()
        user['age'] = int(item['age'])
        user['password_hash'] = hashlib.md5(item['password'].encode()).hexdigest()
        user['is_admin'] = item.get('role', '') == 'admin'
        conn = sqlite3.connect('users.db')
        conn.execute(
            'INSERT INTO users VALUES (?, ?, ?, ?, ?)',
            (user['name'], user['email'], user['age'], user['password_hash'], user['is_admin'])
        )
        conn.commit()
        conn.close()
        users.append(user)
    return users
"""

arena3 = DebateArena(
    f"Review this code from your expert perspective:\n{code_to_review}",
    num_rounds=3
)

arena3.add_debater(DebaterAgent(
    name="PerfReviewer",
    perspective="Performance and efficiency",
    system_prompt=(
        "You are 'PerfReviewer', a performance-focused code reviewer. "
        "Identify performance issues: unnecessary allocations, O(n) problems, "
        "database connection overhead, missing batching, etc. Suggest specific optimizations. "
        "Keep each response to 4-6 sentences."
    )
))

arena3.add_debater(DebaterAgent(
    name="SecReviewer",
    perspective="Security vulnerabilities",
    system_prompt=(
        "You are 'SecReviewer', a security-focused code reviewer. "
        "Identify security vulnerabilities: weak hashing, SQL injection, input validation, "
        "sensitive data handling, authentication issues. Cite OWASP guidelines where relevant. "
        "Keep each response to 4-6 sentences."
    )
))

arena3.add_debater(DebaterAgent(
    name="MaintReviewer",
    perspective="Code maintainability and design",
    system_prompt=(
        "You are 'MaintReviewer', focused on code maintainability. "
        "Identify issues with readability, separation of concerns, error handling, testing, "
        "and modularity. Suggest refactoring patterns and clean code practices. "
        "Keep each response to 4-6 sentences."
    )
))

result_e3 = arena3.run()

## 7. Voting Mechanisms — Alternatives to a Judge

The judge pattern has one LLM call evaluate all arguments. But we can also use **voting** — where each debater votes on the best conclusion.

### 7.1 — Majority Vote, Weighted Vote, Ranked Choice

In [ ]:
class VotingMechanism:
    """Different voting strategies for multi-agent consensus."""

    @staticmethod
    def majority_vote(debaters: List[DebaterAgent], topic: str, options: List[str]) -> Dict:
        """Each debater votes for one option. Majority wins."""
        votes = {}
        for debater in debaters:
            messages = [
                {"role": "system", "content": (
                    f"You are {debater.name}. Given the debate topic and options, "
                    f"vote for the SINGLE best option. Respond with ONLY the option text."
                )},
                {"role": "user", "content": (
                    f"Topic: {topic}\n"
                    f"Options:\n" + "\n".join(f"  {i+1}. {opt}" for i, opt in enumerate(options)) +
                    f"\n\nVote for one option (respond with the number only):"
                )}
            ]
            response = generate(messages, max_new_tokens=50, temperature=0.3)
            # Extract vote
            try:
                vote_idx = int(re.search(r'\d+', response).group()) - 1
                vote = options[vote_idx] if 0 <= vote_idx < len(options) else options[0]
            except (AttributeError, ValueError, IndexError):
                vote = options[0]  # default
            votes[debater.name] = vote

        # Count votes
        vote_counts = Counter(votes.values())
        winner = vote_counts.most_common(1)[0] if vote_counts else (options[0], 0)
        return {
            "method": "majority_vote",
            "votes": votes,
            "counts": dict(vote_counts),
            "winner": winner[0],
            "winner_votes": winner[1],
            "unanimous": len(set(votes.values())) == 1
        }

    @staticmethod
    def weighted_vote(debaters: List[DebaterAgent], topic: str,
                       options: List[str], weights: Dict[str, float]) -> Dict:
        """Each debater votes, but votes are weighted by expertise score."""
        votes = {}
        for debater in debaters:
            messages = [
                {"role": "system", "content": (
                    f"You are {debater.name}. Vote for the best option. "
                    f"Respond with ONLY the option number."
                )},
                {"role": "user", "content": (
                    f"Topic: {topic}\n"
                    f"Options:\n" + "\n".join(f"  {i+1}. {opt}" for i, opt in enumerate(options)) +
                    f"\n\nYour vote (number only):"
                )}
            ]
            response = generate(messages, max_new_tokens=50, temperature=0.3)
            try:
                vote_idx = int(re.search(r'\d+', response).group()) - 1
                vote = options[vote_idx] if 0 <= vote_idx < len(options) else options[0]
            except (AttributeError, ValueError, IndexError):
                vote = options[0]
            votes[debater.name] = vote

        # Apply weights
        weighted_counts = defaultdict(float)
        for debater_name, vote in votes.items():
            weight = weights.get(debater_name, 1.0)
            weighted_counts[vote] += weight

        winner = max(weighted_counts.items(), key=lambda x: x[1])
        return {
            "method": "weighted_vote",
            "votes": votes,
            "weights": weights,
            "weighted_counts": dict(weighted_counts),
            "winner": winner[0],
            "winner_score": round(winner[1], 2)
        }

    @staticmethod
    def ranked_choice(debaters: List[DebaterAgent], topic: str,
                       options: List[str]) -> Dict:
        """Each debater ranks all options. Points: 1st=N, 2nd=N-1, etc."""
        n_options = len(options)
        rankings = {}

        for debater in debaters:
            messages = [
                {"role": "system", "content": (
                    f"You are {debater.name}. Rank all options from best to worst. "
                    f"Respond with option numbers in order, separated by commas (e.g., '2,1,3')."
                )},
                {"role": "user", "content": (
                    f"Topic: {topic}\n"
                    f"Options:\n" + "\n".join(f"  {i+1}. {opt}" for i, opt in enumerate(options)) +
                    f"\n\nRank from best to worst (numbers only, comma-separated):"
                )}
            ]
            response = generate(messages, max_new_tokens=50, temperature=0.3)
            try:
                ranking = [int(x.strip()) - 1 for x in re.findall(r'\d+', response)]
                ranking = [r for r in ranking if 0 <= r < n_options]
                # Fill in any missing options
                for i in range(n_options):
                    if i not in ranking:
                        ranking.append(i)
                rankings[debater.name] = ranking[:n_options]
            except (ValueError, IndexError):
                rankings[debater.name] = list(range(n_options))

        # Compute Borda count scores
        scores = defaultdict(int)
        for debater_name, ranking in rankings.items():
            for rank_pos, option_idx in enumerate(ranking):
                points = n_options - rank_pos
                scores[options[option_idx]] += points

        winner = max(scores.items(), key=lambda x: x[1])
        return {
            "method": "ranked_choice",
            "rankings": {d: [options[i] for i in r] for d, r in rankings.items()},
            "borda_scores": dict(scores),
            "winner": winner[0],
            "winner_score": winner[1]
        }

voting = VotingMechanism()
print("VotingMechanism class ready")

In [ ]:
# Compare voting methods on the nuclear energy debate
nuclear_options = [
    "Nuclear energy should be expanded as a primary climate solution",
    "Nuclear energy should be phased out in favor of renewables",
    "Nuclear energy should complement renewables with modernized technology"
]

# Use the debaters from Experiment 1
debaters = arena1.debaters

print("VOTING COMPARISON: Nuclear Energy & Climate")
print("=" * 70)

# Majority vote
majority = voting.majority_vote(debaters, arena1.topic, nuclear_options)
print(f"\n1. MAJORITY VOTE")
for name, vote in majority['votes'].items():
    print(f"   {name}: {vote[:60]}...")
print(f"   Winner: {majority['winner'][:60]}... ({majority['winner_votes']} votes)")

# Weighted vote (Pragmatist gets higher weight)
weights = {"Pro-Nuclear": 1.0, "Anti-Nuclear": 1.0, "Pragmatist": 1.5}
weighted = voting.weighted_vote(debaters, arena1.topic, nuclear_options, weights)
print(f"\n2. WEIGHTED VOTE (Pragmatist weighted 1.5x)")
print(f"   Weighted scores: {weighted['weighted_counts']}")
print(f"   Winner: {weighted['winner'][:60]}...")

# Ranked choice
ranked = voting.ranked_choice(debaters, arena1.topic, nuclear_options)
print(f"\n3. RANKED CHOICE (Borda count)")
for opt, score in sorted(ranked['borda_scores'].items(), key=lambda x: x[1], reverse=True):
    print(f"   {score} pts: {opt[:60]}...")
print(f"   Winner: {ranked['winner'][:60]}...")

# Judge result
print(f"\n4. JUDGE VERDICT")
print(f"   Winner: {result_e1['winner']}")
print(f"   Conclusion: {result_e1['conclusion'][:150]}...")

## 8. Scaling Analysis — 2 vs 3 vs 5 Debaters

More debaters means more perspectives but also more cost and potentially diminishing returns. Let's measure.

In [ ]:
# Run the same topic with 2, 3, and 5 debaters
scaling_topic = "What is the most effective approach to reducing carbon emissions globally?"

perspectives_pool = [
    ("Renewables", "Rapid expansion of solar and wind energy is the fastest path to decarbonization"),
    ("Nuclear", "Nuclear power provides reliable, carbon-free baseload energy"),
    ("Carbon Tax", "A global carbon tax is the most efficient mechanism to reduce emissions"),
    ("Adaptation", "We should focus on adapting to climate change rather than trying to prevent it"),
    ("Technology", "Carbon capture and geoengineering technologies are essential for net-zero targets"),
]

scaling_results = []

for n_debaters in [2, 3, 5]:
    arena = DebateArena(scaling_topic, num_rounds=2)  # 2 rounds to save time

    for name, perspective in perspectives_pool[:n_debaters]:
        arena.add_debater(DebaterAgent(name=name, perspective=perspective))

    print(f"\n{'='*70}")
    print(f"Running debate with {n_debaters} debaters...")
    result = arena.run(verbose=False)

    scaling_results.append({
        "n_debaters": n_debaters,
        "total_arguments": result['total_arguments'],
        "total_words": result['total_words'],
        "elapsed": result['elapsed_seconds'],
        "winner": result['winner'],
        "conclusion_length": len(result['conclusion']),
        "avg_score": sum(s.overall for s in result['scores']) / len(result['scores'])
    })

    # Print summary
    print(f"  Arguments: {result['total_arguments']}, Words: {result['total_words']}, Time: {result['elapsed_seconds']}s")
    print(f"  Winner: {result['winner']}")
    for s in result['scores']:
        print(f"    {s.debater:15s} overall={s.overall:.2f}")

# Scaling comparison table
print(f"\n\n{'='*70}")
print("SCALING ANALYSIS")
print(f"{'='*70}")
print(f"{'Debaters':>10s} {'Arguments':>10s} {'Words':>8s} {'Time':>8s} {'Avg Score':>10s}")
print("-" * 50)
for r in scaling_results:
    print(f"{r['n_debaters']:>10d} {r['total_arguments']:>10d} {r['total_words']:>8d} "
          f"{r['elapsed']:>7.1f}s {r['avg_score']:>10.2f}")

# Cost scaling
if len(scaling_results) >= 2:
    base_time = scaling_results[0]['elapsed']
    print(f"\nCost scaling (relative to 2 debaters):")
    for r in scaling_results:
        ratio = r['elapsed'] / base_time if base_time > 0 else 0
        print(f"  {r['n_debaters']} debaters: {ratio:.1f}x time")

## 9. Key Takeaways

### What We Built

| Component | Purpose |
|-----------|---------|
| **DebaterAgent** | Argues from assigned perspective with structured claims |
| **JudgeAgent** | Scores arguments on evidence, reasoning, rebuttal quality |
| **DebateArena** | Orchestrates multi-round debates with N debaters |
| **VotingMechanism** | Majority, weighted, and ranked-choice voting |

### Key Insights

1. **Debate improves reasoning quality** — adversarial pressure forces agents to provide better evidence and address counter-arguments
2. **Structured arguments help** — CLAIM/EVIDENCE/REBUTTAL format produces more useful output than free-form dialogue
3. **The judge pattern vs voting** — judge provides nuanced evaluation; voting is simpler and cheaper
4. **Diminishing returns with scale** — 3 debaters is often the sweet spot; 5+ adds cost without proportional quality gains
5. **Perspective assignment matters** — explicitly assigning different viewpoints produces more diverse arguments than hoping for diversity
6. **Code review benefits most** — the three-angle code review (performance, security, maintainability) demonstrates clear value over single-perspective review

### When to Use Debate

| Scenario | Recommended Setup |
|----------|------------------|
| Factual question | 3 debaters + judge, 2-3 rounds |
| Problem-solving | 3 domain experts + judge, 3 rounds |
| Code review | 3 specialized reviewers, 2 rounds |
| Simple factual lookup | Skip debate — single agent is sufficient |

### Design Principles

- **Assign clear, distinct perspectives** — vague perspectives produce similar arguments
- **Limit rounds to 2-3** — quality peaks early, then agents start repeating
- **Use structured output formats** — parse-friendly formats enable automated scoring
- **Log all arguments** — the debate trace is valuable for debugging and auditing

---

**Next Notebook**: [19_sequential_agent_pipelines.ipynb](./19_sequential_agent_pipelines.ipynb) — chaining specialized agents into sequential pipelines.